In [2]:
import pandas   

In [4]:
import os

current_dir = os.getcwd()
print(current_dir)

c:\Users\nzhuw\OneDrive\Desktop\10. Application Development\Diverzify\diverzify_freight_analysis\notebooks\phase II


In [ ]:
partA = pandas.read_excel('data/Part list A with Supplier.xlsx',  sheet_name="Export Worksheet")
partA.head(2)

In [ ]:
partB = pandas.read_excel('data/Part B with Supplier.xlsx', sheet_name="Export Worksheet")

In [ ]:
frames = [partA, partB]
df = pandas.concat(frames, ignore_index=True)

In [ ]:
import re
import matplotlib.pyplot as plt
from collections import Counter

# ——————————————————————————————
# 7. Token-Count Distribution
# ——————————————————————————————
df['token_count'] = (
    df['DESCRIPTION']
    .fillna('')
    .str.split()
    .apply(len)
)
print("Token count stats:")
print(df['token_count'].describe(), "\n")

plt.figure()
df['token_count'].hist(bins=20)
plt.title("Token Count Distribution")
plt.xlabel("Number of tokens")
plt.ylabel("Count")
plt.show()


# ——————————————————————————————
# 8. Vocabulary Size & Top Tokens
# ——————————————————————————————
all_tokens = (
    df['DESCRIPTION']
    .dropna()
    .str.lower()
    .str.findall(r'\b\w+\b')
    .sum()
)
token_counts = Counter(all_tokens)
print(f"Total unique tokens: {len(token_counts)}")
print("Top 20 tokens:", token_counts.most_common(20), "\n")


# ——————————————————————————————
# 9. Separator-Count Distribution
# ——————————————————————————————
seps = ['-', ',', ';', '|', ':']
for sep in seps:
    col = f"sep_count_{sep}"
    df[col] = df['DESCRIPTION'].str.count(re.escape(sep))
    print(f"Average {sep!r} per row: {df[col].mean():.2f}")
print()


# ——————————————————————————————
# 10. Parenthesis & Bracket Usage
# ——————————————————————————————
df['n_parens_open']  = df['DESCRIPTION'].str.count(r'\(')
df['n_parens_close'] = df['DESCRIPTION'].str.count(r'\)')
print("Parenthesis usage stats:")
print(df[['n_parens_open','n_parens_close']].describe(), "\n")


# ——————————————————————————————
# 11. Numeric vs. Alpha-Only Tokens
# ——————————————————————————————
df['numeric_tokens'] = (
    df['DESCRIPTION']
    .str.findall(r'\b\d+[\d/\.]*\b')
    .apply(len)
)
df['alpha_tokens'] = (
    df['DESCRIPTION']
    .str.findall(r'\b[a-zA-Z]+\b')
    .apply(len)
)
print("Numeric vs. alphabetic token counts:")
print(df[['numeric_tokens','alpha_tokens']].describe(), "\n")


# ——————————————————————————————
# 12. Uppercase-Character Ratio
# ——————————————————————————————
def upper_ratio(s):
    if not isinstance(s, str) or not s:
        return 0.0
    uppers = sum(1 for c in s if c.isupper())
    return uppers / len(s)

df['upper_ratio'] = df['DESCRIPTION'].apply(upper_ratio)
print("Uppercase character ratio stats:")
print(df['upper_ratio'].describe(), "\n")


# ——————————————————————————————
# 13. Average Token Length
# ——————————————————————————————
def avg_tok_len(s):
    toks = re.findall(r'\b\w+\b', str(s))
    return sum(len(t) for t in toks) / len(toks) if toks else 0.0

df['avg_token_len'] = df['DESCRIPTION'].apply(avg_tok_len)
print("Average token length stats:")
print(df['avg_token_len'].describe(), "\n")


# ——————————————————————————————
# 14. Duplicate & Missing Descriptions
# ——————————————————————————————
n_missing = df['DESCRIPTION'].isna().sum()
n_dups    = df.duplicated('DESCRIPTION').sum()
print(f"Missing descriptions: {n_missing}")
print(f"Exact duplicate rows: {n_dups}")
